# Importing Dependencies

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import math

# Basic Decoder Class Definition with No Expanded Attention
Below is the defintion of our decoder class with no expansion implemented that will be used to generate chess moves. This decoder uses PyTorch's TransformerDecoder that incorporates multi-head self attention and feedforward neural nets, while adding and normalizing after each layer. Postional embeddings are calculated before being fed into the decoder layer. A fully connected layer is used for the output to calculate a softmax for the most probable chess moves.

In [4]:
class BasicChessDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, max_len=200):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=1024)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x shape: [batch, seq_len] → transformer expects [seq_len, batch]
        x = x.transpose(0, 1)
        seq_len, batch_size = x.size()

        # Calculate positional embeddings
        positions = torch.arange(seq_len, device=x.device).unsqueeze(1)
        x = self.embed(x) + self.pos_embed(positions)

        # Decoder masking: prevent attention to future tokens
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()

        x = self.decoder(x, x, tgt_mask=mask)
        logits = self.fc_out(x)  # [seq_len, batch, vocab_size]
        return logits.transpose(0, 1)  # [batch, seq_len, vocab_size]


# Modified Chess Decoder with Multi-Query Attention and its Components
The next four cells contain the components of the expanded decoder model with the class defintion of the decoder below combining all of the components. The components include a multi-head attention class that uses queries, keys, and values to calculate dot products, a feed forward class made up of two linear layers and a relu function, and finally a decoder layer class that combines the other two components and implements normalization and dropout in between. Within the multi-head attention class, the expansion mechanism of multi-query attention, where instead of making new keys and values for each head, we are using the same keys and values across all heads. Much of the basic functionality for the components has been taken from https://www.datacamp.com/tutorial/building-a-transformer-with-py-torch

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value
        
        # Queries: one projection per head
        self.W_q = nn.Linear(d_model, d_model)
        # Shared keys and values across all heads
        self.W_k = nn.Linear(d_model, self.d_k)
        self.W_v = nn.Linear(d_model, self.d_k)

        # Output projection
        self.W_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        # Multi-head queries
        Q = self.split_heads(self.W_q(Q))
        # Shared K, V
        K = self.W_k(K).unsqueeze(1)  # [B, 1, L, d_k]
        V = self.W_v(V).unsqueeze(1)  # [B, 1, L, d_k]
        
        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [6]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [7]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [8]:
class ChessDecoderWithExpansion(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, max_len=200, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads=nhead, d_ff=1024, dropout=dropout) for _ in range(num_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)

        # Calculate positional embeddings
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embed(x) + self.pos_embed(positions)

        # Decoder masking: prevent attention to future tokens
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        mask = mask.unsqueeze(0).unsqueeze(0)  # shape: [1, 1, seq_len, seq_len]


        for dec_layer in self.decoder_layers:
            x = dec_layer(x, mask)
    
        logits = self.fc_out(x)
        return logits  # [batch, seq_len, vocab_size]


Pulling in the .pt file that contains the tensor of all encoded chess games that will be used for training.

In [9]:
encoded_tensor = torch.load("encoded_games_small.pt")

print(type(encoded_tensor))
print(encoded_tensor.shape)
print(encoded_tensor[0][:10])  # first few move IDs of first game


<class 'torch.Tensor'>
torch.Size([49854, 200])
tensor([    1, 10536, 10644, 10609, 10542, 10683,   303,  1791,  1347,  1338])


Defining the dataset we will use during training to store and pull game moves. Also intializing dataset and data loader with tensor containing the encoded chess games.

In [10]:
class ChessDataset(Dataset):
    def __init__(self, encoded_tensor):
        self.data = encoded_tensor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx][:-1]  # all but last
        y = self.data[idx][1:]   # all but first
        return x, y

dataset = ChessDataset(encoded_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)


The next few cells are used to test the funtionality of the loader and model when working together.

In [11]:
with open("move_to_id.json", "r") as f:
    move_to_id = json.load(f)

vocab_size = len(move_to_id)
model = ChessDecoderWithExpansion(vocab_size=vocab_size)

In [12]:
x, y = next(iter(loader))

print("Input batch shape:", x.shape)
print("Target batch shape:", y.shape)
print("Example input sequence:", x[0][:10])
print("Example target sequence:", y[0][:10])

Input batch shape: torch.Size([64, 199])
Target batch shape: torch.Size([64, 199])
Example input sequence: tensor([    1, 10536, 10539,  1791, 10439, 10433, 10647,  1338,  1490, 10851])
Example target sequence: tensor([10536, 10539,  1791, 10439, 10433, 10647,  1338,  1490, 10851,  1987])


In [13]:
with torch.no_grad():
    logits = model(x)
print("Model output shape:", logits.shape)

Model output shape: torch.Size([64, 199, 11017])


In [14]:
PAD_ID = move_to_id['<PAD>']
import torch.nn.functional as F

loss = F.cross_entropy(
    logits.reshape(-1, vocab_size),
    y.reshape(-1),
    ignore_index=PAD_ID
)
print("Test loss:", loss.item())


Test loss: 9.463298797607422


# Hyperparameter Optimization
We will use optuna's functionality on our datadset to tune hyperparameters before training on our larger dataset.

In [20]:
from torch.utils.data import TensorDataset, random_split, DataLoader

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

import optuna
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device {device} found')
vocab_size = len(move_to_id)

def objective(trial):
    # search space
    d_model = trial.suggest_categorical('d_model', [128, 256, 512])
    nhead = trial.suggest_categorical('nhead', [4,8])
    num_layers = trial.suggest_int('num_layers', 2, 6)
    lr = trial.suggest_float('lr', 1e-4, 5e-3)

    # create model
    model = ChessDecoderWithExpansion(
        vocab_size=vocab_size,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        max_len=encoded_tensor.size(1)
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    # train a few epochs
    EPOCHS = 3
    for epoch in range(EPOCHS):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
            loss.backward()
            optimizer.step()

    # validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for x, y, in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits.reshape(-1,vocab_size), y.reshape(-1))
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    return avg_val_loss


device cuda found


run optimization with optuna

In [21]:
# set up optuna

from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# run a few rounds at random first
sampler = TPESampler(n_startup_trials=3)
pruner = MedianPruner(n_startup_trials=3, n_warmup_steps=1)

study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=10)

# print results

print('\n\nBest trial:')
print('  Value (val loss):', study.best_trial.value)
print('  Params:', study.best_trial.params)

[I 2025-10-26 16:38:35,718] A new study created in memory with name: no-name-9ef4022e-3685-467c-8abb-0c89fa7f425b
[I 2025-10-26 16:40:38,514] Trial 0 finished with value: 5.720999950017685 and parameters: {'d_model': 512, 'nhead': 4, 'num_layers': 2, 'lr': 0.003352496531528272}. Best is trial 0 with value: 5.720999950017685.
[I 2025-10-26 16:44:04,872] Trial 1 finished with value: 0.007351691258521989 and parameters: {'d_model': 512, 'nhead': 4, 'num_layers': 5, 'lr': 0.0004991511603841531}. Best is trial 1 with value: 0.007351691258521989.
[I 2025-10-26 16:45:27,563] Trial 2 finished with value: 0.15687201558970487 and parameters: {'d_model': 256, 'nhead': 4, 'num_layers': 2, 'lr': 0.004498587732961306}. Best is trial 1 with value: 0.007351691258521989.
[I 2025-10-26 16:49:39,309] Trial 3 finished with value: 0.01665910347424543 and parameters: {'d_model': 512, 'nhead': 8, 'num_layers': 6, 'lr': 0.000160890910973996}. Best is trial 1 with value: 0.007351691258521989.
[I 2025-10-26 16:



Best trial:
  Value (val loss): 0.007351691258521989
  Params: {'d_model': 512, 'nhead': 4, 'num_layers': 5, 'lr': 0.0004991511603841531}


The following cell allows for easy storage and retrieval of the best hyperparameters as found above

In [22]:
# save best hyperparameters to file

import json
from datetime import datetime

best_params = study.best_trial.params
best_value = study.best_trial.value

best_params_with_meta = {
    "best_params": best_params,
    "best_val_loss": best_value,
    "n_trials": len(study.trials),
}

filename = f'best_hparams_expanded.json'

with open(filename, 'w') as f:
    json.dump(best_params_with_meta, f)


# how to open later or in different file
'''
with open("best_hparams.json", "r") as f:
    best_hparams = json.load(f)

params = best_hparams["best_params"]

best_model = ChessDecoder(
    vocab_size=vocab_size,
    d_model=params["d_model"],
    nhead=params["nhead"],
    num_layers=params["num_layers"],
    max_len=encoded_tensor.size(1)
).to(device)
'''

'\nwith open("best_hparams.json", "r") as f:\n    best_hparams = json.load(f)\n\nparams = best_hparams["best_params"]\n\nbest_model = ChessDecoder(\n    vocab_size=vocab_size,\n    d_model=params["d_model"],\n    nhead=params["nhead"],\n    num_layers=params["num_layers"],\n    max_len=encoded_tensor.size(1)\n).to(device)\n'

# Model Training
Training is done after the best combination of hyperparameters have been found. These hyperparameters are used to create the final instance of our model. This model is trained on an encoded dataset of 1 million chess games and uses cross entropy to calcuate loss.

In [25]:
training_tensor = encoded_tensor[:40000]

dataset = ChessDataset(training_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim

vocab_size = len(move_to_id)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'On device {device}')

with open("best_hparams_expanded.json", "r") as f:
    best_hparams = json.load(f)

params = best_hparams["best_params"]

model = ChessDecoderWithExpansion(
    vocab_size=vocab_size,
    d_model=params["d_model"],
    nhead=params["nhead"],
    num_layers=params["num_layers"],
    max_len=encoded_tensor.size(1)
).to(device)

# For resuming training
#model.load_state_dict(torch.load('standard_model.pt', map_location=device))
#model.train()

optimizer = optim.Adam(model.parameters(), lr=params["lr"])

epoch_loss = 0
num_batches = 0
for epoch in range(50):
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        
        logits = model(x)
        loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            y.reshape(-1),
            ignore_index=PAD_ID
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    torch.save(model.state_dict(), 'expanded_model.pt')
    print(f"Epoch {epoch+1} average loss = {epoch_loss / num_batches:.4f}")
 


On device cuda
Epoch 1 average loss = 0.8438
Epoch 2 average loss = 0.4281
Epoch 3 average loss = 0.2869
Epoch 4 average loss = 0.2157
Epoch 5 average loss = 0.1728
Epoch 6 average loss = 0.1443
Epoch 7 average loss = 0.1238
Epoch 8 average loss = 0.1084
Epoch 9 average loss = 0.0965
Epoch 10 average loss = 0.0870
Epoch 11 average loss = 0.0791
Epoch 12 average loss = 0.0726
Epoch 13 average loss = 0.0671
Epoch 14 average loss = 0.0624
Epoch 15 average loss = 0.0582
Epoch 16 average loss = 0.0547
Epoch 17 average loss = 0.0515
Epoch 18 average loss = 0.0487
Epoch 19 average loss = 0.0462
Epoch 20 average loss = 0.0439
Epoch 21 average loss = 0.2160
Epoch 22 average loss = 0.4915
Epoch 23 average loss = 0.7301
Epoch 24 average loss = 0.9449
Epoch 25 average loss = 1.1421
Epoch 26 average loss = 1.3230
Epoch 27 average loss = 1.4892
Epoch 28 average loss = 1.6414
Epoch 29 average loss = 1.7856
Epoch 30 average loss = 1.9195
Epoch 31 average loss = 2.0432
Epoch 32 average loss = 2.1603
Ep